# Pipeline Inspection: Inference Pipeline (TomoSAR Full-Scene Reconstruction)

This notebook executes the `InferencePipeline` (`pipelines/inference_pipeline/pipeline.py`) stage by stage and verifies that each step produces the contracted output before the next stage consumes it.

The pipeline loads a trained per-pixel Gaussian-mixture model from a completed training run, predicts Gaussian parameters for every patch of a held-out split of a tomographic SAR scene, reconstructs the elevation intensity profile at each pixel by summing the predicted Gaussians, stitches the overlapping patch cubes into a single full-scene reconstruction using a windowed overlap-add blend, evaluates the prediction against the ground-truth Gaussian reconstruction, renders publication-quality diagnostic figures and animations, and writes a Markdown report.

**What this notebook verifies, stage by stage:**

- The trained checkpoint, EMA shadow weights, normalization statistics and patch grid are loaded consistently and the model is in `eval()` mode.
- The per-patch forward pass returns denormalized Gaussian parameters in physical units.
- The Gaussian superposition reconstructs elevation profiles of the expected shape and range.
- The windowed overlap-add stitcher reassembles the full scene with no unweighted seams.
- The evaluation metrics (curve MSE/MAE/RMSE/R2/PSNR, per-pixel maps, SSIM, per-elevation-bin curves, Gaussian parameter errors, placeholder detection) are finite and within plausible bounds.
- The figures, animations and report are written to the expected paths.

> Execute top to bottom. The notebook assumes a completed training run directory is available at the path set in the configuration cell below.

In [ ]:
from __future__ import annotations

import sys
import json
from pathlib import Path

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from configuration.inference_config import InferenceConfig
from pipelines.inference_pipeline.pipeline import InferencePipeline
from pipelines.inference_pipeline.loader import RunLoader, Run
from pipelines.inference_pipeline.predictor import Predictor
from pipelines.inference_pipeline.reconstruction import GaussianReconstructor
from pipelines.inference_pipeline.stitching import CubeStitcher
from pipelines.inference_pipeline.metrics import Metrics
from pipelines.inference_pipeline.metadata import InferenceMetadata
from pipelines.inference_pipeline.figures import FigureComposer
from pipelines.inference_pipeline.plots import Ploter
from pipelines.inference_pipeline.report import Report, ReportPayloadBuilder
from pipelines.inference_pipeline.types import Result
from tools.logger import Logger

mpl.rcParams.update({
    "font.family"        : "serif",
    "font.size"          : 11,
    "axes.labelsize"     : 12,
    "axes.titlesize"     : 13,
    "legend.fontsize"    : 10,
    "xtick.labelsize"    : 10,
    "ytick.labelsize"    : 10,
    "figure.dpi"         : 150,
    "savefig.dpi"        : 300,
    "savefig.bbox"       : "tight",
    "axes.spines.top"    : False,
    "axes.spines.right"  : False,
})

figure_directory = Path("figures/inspect_inference_pipeline")
figure_directory.mkdir(parents=True, exist_ok=True)

inference_configuration = InferenceConfig(
    run_directory = project_root / "runs" / "latest_run",
    output_subdir = "notebook_inspection",
    device        = "cuda",
    use_ema       = True,
    split         = "test",
)

print("Project root      :", project_root)
print("Run directory     :", inference_configuration.run_directory)
print("Split             :", inference_configuration.split)
print("Device            :", inference_configuration.device)
print("Use EMA           :", inference_configuration.use_ema)
print("Stitch window     :", inference_configuration.stitch_window)
print("Figure directory  :", figure_directory.resolve())

## Pipeline Overview

`InferencePipeline.run()` orchestrates the following ordered sequence. Each arrow is a handoff of a concrete object from one component to the next.

```
InferenceConfig
  └─► Stage 1  _setup()              →  InferenceMetadata + Logger + Ploter   (dirs, seed)
        └─► Stage 2  RunLoader.load() →  Run  (model, EMA, normalizer, dataset, loader, grid)
              └─► Stage 3  ModelWrapper →  callable that denormalizes + clamps model output
                    └─► Stage 4  Predictor._forward_pass()  →  per-patch denorm Gaussian params
                          └─► Stage 5  GaussianReconstructor + _cpu_worker →  per-patch curves + metrics
                                └─► Stage 6  CubeStitcher / _stitch_results →  full-scene cubes (Result)
                                      └─► Stage 7  _compute_slice_indices()  →  slice index dict
                                            └─► Stage 8  Metrics.compute()    →  global_metrics + metrics.json
                                                  └─► Stage 9  FigureComposer.compose() →  figure_paths
                                                        └─► Stage 10 FigureComposer.animate() → gif_paths
                                                              └─► Stage 11 Report.assemble() → report.md (Path)
```

| Stage | Callable | Input | Output |
|-------|----------|-------|--------|
| 1 | `InferencePipeline._setup(config)` | `InferenceConfig` | `(InferenceMetadata, Logger, Ploter)` |
| 2 | `RunLoader.load(...)` | run directory, split, device, EMA flag | `Run` dataclass |
| 3 | `ModelWrapper.__call__` / EMA application | trained `state_dict`, EMA shadow, norm `Stats` | denormalizing callable model |
| 4 | `Predictor._forward_pass()` | `Run.loader` (patch tensors) | lists of denorm patch params (pred + GT) |
| 5 | `Predictor._compute_metrics()` / `GaussianReconstructor.reconstruct_batch` | patch params, `x_axis` | per-patch curves + per-pixel metric maps |
| 6 | `Predictor._stitch_results()` / `CubeStitcher` | per-patch curves, `GridInfo` | full-scene `Result` (cubes + maps) |
| 7 | `InferencePipeline._compute_slice_indices(...)` | cube shape, slice counts | dict of slice index arrays |
| 8 | `Metrics.compute(...)` | `Result`, `x_axis`, slice indices | `global_metrics` dict + `metrics.json` |
| 9 | `FigureComposer.compose(...)` | `Result`, `global_metrics`, indices | `Dict[str, Path]` figure paths |
| 10 | `FigureComposer.animate(...)` | `Result`, `x_axis` | `Dict[str, Path]` GIF paths |
| 11 | `InferencePipeline._build_report(...)` / `Report.assemble()` | all of the above | `report.md` `Path` |

**Domain context.** A TomoSAR scene is indexed by (azimuth, range) on the ground and by elevation along the line of sight. For every (azimuth, range) pixel the model predicts $K$ Gaussian components $\{(a_k, \mu_k, \sigma_k)\}$ describing the vertical reflectivity profile. The reconstructed elevation profile (tomogram column) is the superposition
$$ \hat{y}(x_n) = \sum_{k=1}^{K} a_k \exp\!\left( -\frac{(x_n - \mu_k)^2}{2\sigma_k^2 + \epsilon} \right). $$
The scene is tiled into overlapping patches; reconstruction is per patch and the patches are blended back into one full-scene cube of shape `(N_elevation, azimuth, range)`.

---
## Stage 1: Setup — metadata, directories, seed, logger, plotter

**Callable:** `InferencePipeline._setup(config)`
**Input:** the `InferenceConfig` instance.
**Output:** `(InferenceMetadata, Logger, Ploter)`. Side effects: creates the output / figures / animations / logs / cubes directories and seeds NumPy with `config.seed`.

No mathematical transformation; this stage establishes the deterministic, reproducible environment every later stage relies on. The output directory is `run_directory / 'inference' / output_subdir`.

> **What you should see:** an `InferenceMetadata` whose `output_dir`, `figures_dir`, `animations_dir`, `logs_dir` and `cube_dir` all exist on disk and all sit under `run_directory / 'inference'`. `metrics_path` ends with `metrics.json` and `report_path` ends with `report.md`. The `Ploter` carries the configured intensity colormap (`'jet'`), error colormap (`'magma'`), `normalize=True`, and a save DPI of 300. The NumPy global seed is set to `config.seed` (0).

In [ ]:
inference_metadata = InferenceMetadata(inference_configuration)
inference_metadata.create_dirs()
np.random.seed(inference_configuration.seed)

inference_logger = Logger(
    log_dir = str(inference_metadata.logs_dir),
    name    = "inference",
    level   = inference_configuration.log_level,
)

inference_plotter = Ploter(
    cmap      = inference_configuration.cmap_intensity,
    err_cmap  = inference_configuration.cmap_error,
    normalize = inference_configuration.normalize_intensity,
    fig_dpi   = inference_configuration.fig_dpi,
    save_dpi  = inference_configuration.save_dpi,
)

In [ ]:
print("output_dir       :", inference_metadata.output_dir)
print("figures_dir      :", inference_metadata.figures_dir)
print("animations_dir   :", inference_metadata.animations_dir)
print("logs_dir         :", inference_metadata.logs_dir)
print("cube_dir         :", inference_metadata.cube_dir)
print("metrics_path     :", inference_metadata.metrics_path)
print("report_path      :", inference_metadata.report_path)
print()
print("directory existence:")
for directory_label, directory_path in (
    ("output_dir",     inference_metadata.output_dir),
    ("figures_dir",    inference_metadata.figures_dir),
    ("animations_dir", inference_metadata.animations_dir),
    ("logs_dir",       inference_metadata.logs_dir),
    ("cube_dir",       inference_metadata.cube_dir),
):
    print(f"  {directory_label:<16}: exists={directory_path.exists()}")
print()
print("plotter cmap     :", inference_plotter.cmap)
print("plotter err_cmap :", inference_plotter.err_cmap)
print("plotter normalize:", inference_plotter.normalize)
print("plotter save_dpi :", inference_plotter.save_dpi)

In [ ]:
subdirectory_labels = ["output", "figures", "animations", "logs", "cubes"]
subdirectory_exists = [
    inference_metadata.output_dir.exists(),
    inference_metadata.figures_dir.exists(),
    inference_metadata.animations_dir.exists(),
    inference_metadata.logs_dir.exists(),
    inference_metadata.cube_dir.exists(),
]

figure, axis = plt.subplots(figsize=(7, 3.2))
bar_colors   = ["#2a7f62" if present else "#b03030" for present in subdirectory_exists]
axis.bar(subdirectory_labels, [1.0 if present else 0.0 for present in subdirectory_exists], color=bar_colors, edgecolor="black", linewidth=0.6)
axis.set_ylim(0, 1.2)
axis.set_ylabel("created (1) / missing (0)")
axis.set_xlabel("output sub-directory")
axis.set_title("Stage 1 — Setup: output directory creation status")
axis.set_yticks([0, 1])
figure.savefig(figure_directory / "stage_1_setup.pdf")
plt.show()

In [ ]:
stage_1_checks = [
    ("output_dir created",                inference_metadata.output_dir.exists()),
    ("figures_dir created",               inference_metadata.figures_dir.exists()),
    ("animations_dir created",            inference_metadata.animations_dir.exists()),
    ("logs_dir created",                  inference_metadata.logs_dir.exists()),
    ("cube_dir created",                  inference_metadata.cube_dir.exists()),
    ("output_dir under run/inference",    (inference_configuration.run_directory / "inference") in inference_metadata.output_dir.parents),
    ("metrics_path name matches config",  inference_metadata.metrics_path.name == inference_configuration.paths.metrics_filename),
    ("report_path name matches config",   inference_metadata.report_path.name == inference_configuration.paths.report_filename),
    ("plotter cmap matches config",       inference_plotter.cmap == inference_configuration.cmap_intensity),
    ("plotter save_dpi matches config",   inference_plotter.save_dpi == inference_configuration.save_dpi),
]
for stage_1_description, stage_1_condition in stage_1_checks:
    print(f"[{'PASS' if stage_1_condition else 'FAIL'}] {stage_1_description}")

### Common Mistakes — Stage 1: Setup

| Symptom | Likely Cause | How to Diagnose |
|---------|--------------|-----------------|
| Outputs overwrite a previous run | `output_subdir` reused across runs instead of the timestamp default | Print `inference_metadata.output_dir`; leave `output_subdir=None` to get a `YYYYMMDD_HHMMSS` folder |
| Figures saved but cube `.npy` files missing | `cube_dir` not created before the stitcher writes its memmap | Check `inference_metadata.cube_dir.exists()` here, before Stage 6 |
| Non-reproducible profile / scatter sampling downstream | `np.random.seed(config.seed)` skipped or a different seed used | Confirm the seed line ran; `profile_seed` also drives pixel selection in Stage 9 |
| Plots ignore the configured colormap or DPI | `Ploter` constructed with hardcoded args instead of config fields | Compare `inference_plotter.cmap` / `.save_dpi` against the config |
| `PermissionError` creating directories | `run_directory` points at a read-only or non-existent training run | Verify `inference_configuration.run_directory` exists and is writable |

**Passing to Stage 2:** `inference_metadata` — `InferenceMetadata`, the resolved output paths; `inference_logger` — `Logger`, structured telemetry sink; `inference_plotter` — `Ploter`, the publication-style figure renderer.

---
## Stage 2: Load Run — model, EMA, normalizer, dataset, loader, patch grid

**Callable:** `RunLoader(run_directory, logger).load(split, batch_size, num_workers, device, use_ema, checkpoint_name)`
**Input:** the training run directory and inference options.
**Output:** a `Run` dataclass bundling the trained model (wrapped, in `eval()` mode, optionally with EMA shadow weights applied), the `x_axis` elevation sample points, `n_gaussians = out_channels // 3`, the `DatasetConfiguration`, the chosen split `CropRegion`, the patch `GridInfo`, the `PatchDataset`, the `DataLoader`, and `checkpoint_meta`.

The number of Gaussian components is recovered from the saved head width: $K = C_{\text{out}} / 3$. EMA shadow weights, when present and `use_ema=True`, replace the raw parameters in place: $\theta \leftarrow \tilde\theta$.

> **What you should see:** `run.model_name` is one of the supported architectures; `run.out_channels == 3 * run.n_gaussians`; `run.x_axis` is a 1-D `float32` array of length `run.x_axis_length`, sorted or sortable, spanning the scene's elevation extent in metres; `run.grid.number_of_patches == run.grid.n_v * run.grid.n_h`; the loader yields batches whose image tensor has `run.in_channels` channels; and `run.used_ema` matches whether shadow weights were actually found and applied.

In [ ]:
run_loader = RunLoader(inference_configuration.run_directory, logger=inference_logger)

loaded_run = run_loader.load(
    split           = inference_configuration.split,
    batch_size      = inference_configuration.batch_size,
    num_workers     = inference_configuration.num_workers,
    device          = inference_configuration.device,
    use_ema         = inference_configuration.use_ema,
    checkpoint_name = inference_configuration.checkpoint_name,
)

In [ ]:
print("model_name        :", loaded_run.model_name)
print("in_channels       :", loaded_run.in_channels)
print("out_channels      :", loaded_run.out_channels)
print("n_gaussians (K)   :", loaded_run.n_gaussians)
print("used_ema          :", loaded_run.used_ema)
print()
print("x_axis dtype      :", loaded_run.x_axis.dtype)
print("x_axis length     :", loaded_run.x_axis_length)
print("x_axis min / max  :", float(loaded_run.x_axis.min()), float(loaded_run.x_axis.max()))
print("x_axis step (med) :", float(np.median(np.diff(np.sort(loaded_run.x_axis)))))
print()
print("split_name        :", loaded_run.split_name)
print("split_region      :", loaded_run.split_region.as_tuple())
print("global_crop       :", loaded_run.global_crop.as_tuple())
print()
print("grid n_v x n_h    :", loaded_run.grid.n_v, "x", loaded_run.grid.n_h)
print("grid patches      :", loaded_run.grid.number_of_patches)
print("grid patch_size   :", loaded_run.grid.patch_size)
print("grid stride       :", loaded_run.grid.stride)
print("grid spatial_size :", loaded_run.grid.spatial_size)
print("grid padded_size  :", loaded_run.grid.padded_size)
print("grid padding (t,b,l,r):", loaded_run.grid.pad_top, loaded_run.grid.pad_bot, loaded_run.grid.pad_left, loaded_run.grid.pad_right)
print()
print("dataset length    :", len(loaded_run.dataset))
print("loader batch_size :", loaded_run.dataset_config.batch_size)
print("checkpoint_meta   :", loaded_run.checkpoint_meta)

In [ ]:
elevation_axis_sorted = np.sort(loaded_run.x_axis)
elevation_spacing     = np.diff(elevation_axis_sorted)

figure, (axis_left, axis_right) = plt.subplots(1, 2, figsize=(12, 4))

axis_left.plot(np.arange(loaded_run.x_axis_length), loaded_run.x_axis, color="C0", linewidth=1.0)
axis_left.set_xlabel("elevation sample index")
axis_left.set_ylabel("elevation [m]")
axis_left.set_title("Elevation axis sample points")
axis_left.grid(True, linewidth=0.3, alpha=0.4)

axis_right.hist(elevation_spacing, bins=40, color="C0", edgecolor="white", linewidth=0.3)
axis_right.axvline(float(np.median(elevation_spacing)), color="black", linestyle="--", linewidth=1.0, label=f"median = {np.median(elevation_spacing):.4g} m")
axis_right.set_xlabel("elevation step between consecutive samples [m]")
axis_right.set_ylabel("count")
axis_right.set_title("Elevation sampling uniformity")
axis_right.legend()
axis_right.grid(True, linewidth=0.3, alpha=0.4)

figure.suptitle("Stage 2 — Load Run: elevation axis structure")
figure.tight_layout(rect=(0, 0, 1, 0.96))
figure.savefig(figure_directory / "stage_2_load_run.pdf")
plt.show()

In [ ]:
supported_model_names = {"unet", "resunet", "attention_unet", "unetplusplus", "fcn", "linknet", "swin_unet", "transunet", "unetr", "unet_multihead", "unet_pergaussian"}
model_is_in_eval_mode = (not loaded_run.model._model.training) if hasattr(loaded_run.model, "_model") else True

stage_2_checks = [
    ("model_name is supported",              loaded_run.model_name in supported_model_names),
    ("out_channels == 3 * n_gaussians",      loaded_run.out_channels == 3 * loaded_run.n_gaussians),
    ("n_gaussians >= 1",                     loaded_run.n_gaussians >= 1),
    ("x_axis is 1-D",                        loaded_run.x_axis.ndim == 1),
    ("x_axis length matches x_axis_length",  loaded_run.x_axis.size == loaded_run.x_axis_length),
    ("x_axis is finite",                     bool(np.isfinite(loaded_run.x_axis).all())),
    ("grid patches == n_v * n_h",            loaded_run.grid.number_of_patches == loaded_run.grid.n_v * loaded_run.grid.n_h),
    ("grid stride <= patch height",          loaded_run.grid.stride <= loaded_run.grid.patch_size[0]),
    ("dataset non-empty",                    len(loaded_run.dataset) > 0),
    ("dataset length == grid patches",       len(loaded_run.dataset) == loaded_run.grid.number_of_patches),
    ("model in eval mode",                   model_is_in_eval_mode),
    ("used_ema matches request",             (loaded_run.used_ema is True) or (inference_configuration.use_ema is False) or (loaded_run.used_ema is False)),
]
for stage_2_description, stage_2_condition in stage_2_checks:
    print(f"[{'PASS' if stage_2_condition else 'FAIL'}] {stage_2_description}")

### Common Mistakes — Stage 2: Load Run

| Symptom | Likely Cause | How to Diagnose |
|---------|--------------|-----------------|
| `used_ema=False` despite `use_ema=True` | Checkpoint has an empty or missing `ema_shadow.shadow` dict | Inspect `torch.load(ckpt)['ema_shadow']`; EMA weights vs raw weights change the prediction materially |
| Predictions look untrained / random | `model.eval()` not called, so dropout / BatchNorm run in train mode | Check `loaded_run.model._model.training` is `False` |
| `n_gaussians` wrong, params reshape fails downstream | `out_channels` includes a noise head channel (`3K+1`) but `K=out//3` truncates it | Confirm `out_channels % 3 == 0`; a noise head would break this contract |
| Stitched scene wrong size / shifted | `split_region` offsets or `GridInfo` padding mismatched between train and inference | Compare `split_region.as_tuple()` and `grid.padded_size` with the training metadata |
| Patches shuffled, stitching scrambled | `DataLoader` built with `shuffle=True` | The loader is built with `shuffle=False`; verify if you customise it |
| Normalization stats from a different run | `Stats.load` reads `meta/` of the wrong run directory | Print the loaded `loc`/`scale` and compare to the training run's `meta` |

**Passing to Stage 3:** `loaded_run` — `Run`, holding the wrapped trained model (`loaded_run.model`), the normalizer (`loaded_run.dataset.normalizer`), the patch `DataLoader` (`loaded_run.loader`), and the patch grid (`loaded_run.grid`).

---
## Stage 3: Model Wrapper — denormalization and physical-unit clamping

**Callable:** `ModelWrapper.denormalize_output(out)` (invoked inside `ModelWrapper.__call__`).
**Input:** the raw network output tensor in normalized parameter space, shape `(B, 3K, H, W)`.
**Output:** the same tensor mapped back to physical Gaussian-parameter units and clamped to valid ranges.

Denormalization inverts the per-channel standardization applied during training, $p = \text{scale} \cdot \hat p + \text{loc}$, then `clamp_gaussian_params` constrains each triple: amplitude to $[0, a_{\max}]$, mean $\mu$ to the elevation axis extent $[\min x, \max x]$, and $\sigma$ to a positive range. This is the single most error-prone interface in inference: if the training-time normalization statistics are not re-applied (or are applied in the wrong direction) every downstream metric is meaningless.

> **What you should see:** the normalizer's output `loc` and `scale` arrays have length `3 * run.n_gaussians`; a synthetic normalized input of zeros denormalizes to approximately the `loc` vector (then clamped); after clamping, amplitudes are $\ge 0$, means lie within the elevation axis range, and standard deviations are strictly positive.

In [ ]:
model_wrapper        = loaded_run.model
output_normalization = loaded_run.dataset.normalizer.stats.output_stats

normalization_location = np.asarray(output_normalization.loc,   dtype=np.float64)
normalization_scale    = np.asarray(output_normalization.scale, dtype=np.float64)

import torch

synthetic_zero_input = torch.zeros((1, 3 * loaded_run.n_gaussians, 4, 4), dtype=torch.float32)
denormalized_zero    = model_wrapper.denormalize_output(synthetic_zero_input).cpu().numpy()

In [ ]:
print("output loc   length :", normalization_location.size)
print("output scale length :", normalization_scale.size)
print("expected (3K)       :", 3 * loaded_run.n_gaussians)
print()
print("loc   per channel   :", np.round(normalization_location, 4))
print("scale per channel   :", np.round(normalization_scale, 4))
print()
print("denormalized-zero shape :", denormalized_zero.shape)
for gaussian_index in range(loaded_run.n_gaussians):
    amplitude_channel = denormalized_zero[0, 3 * gaussian_index + 0]
    mean_channel      = denormalized_zero[0, 3 * gaussian_index + 1]
    sigma_channel     = denormalized_zero[0, 3 * gaussian_index + 2]
    print(f"  g{gaussian_index + 1}: a~{amplitude_channel.mean():.4g}  mu~{mean_channel.mean():.4g}  sigma~{sigma_channel.mean():.4g}")

In [ ]:
channel_indices = np.arange(3 * loaded_run.n_gaussians)
channel_kinds   = [("a", "C0"), ("mu", "C1"), ("sigma", "C2")]
channel_colors  = [channel_kinds[index % 3][1] for index in channel_indices]

figure, (axis_left, axis_right) = plt.subplots(1, 2, figsize=(12, 4))

axis_left.bar(channel_indices, normalization_location, color=channel_colors, edgecolor="black", linewidth=0.5)
axis_left.set_xlabel("output channel index (a, mu, sigma per Gaussian)")
axis_left.set_ylabel("location (mean used to de-standardize)")
axis_left.set_title("Output normalization: location per channel")
axis_left.grid(True, axis="y", linewidth=0.3, alpha=0.4)

axis_right.bar(channel_indices, normalization_scale, color=channel_colors, edgecolor="black", linewidth=0.5)
axis_right.set_xlabel("output channel index (a, mu, sigma per Gaussian)")
axis_right.set_ylabel("scale (std used to de-standardize)")
axis_right.set_title("Output normalization: scale per channel")
axis_right.grid(True, axis="y", linewidth=0.3, alpha=0.4)

figure.suptitle("Stage 3 — Model Wrapper: de-normalization statistics")
figure.tight_layout(rect=(0, 0, 1, 0.96))
figure.savefig(figure_directory / "stage_3_model_wrapper.pdf")
plt.show()

In [ ]:
amplitude_channels = denormalized_zero[0, 0::3]
mean_channels      = denormalized_zero[0, 1::3]
sigma_channels     = denormalized_zero[0, 2::3]

elevation_minimum = float(loaded_run.x_axis.min())
elevation_maximum = float(loaded_run.x_axis.max())

stage_3_checks = [
    ("loc length == 3K",                    normalization_location.size == 3 * loaded_run.n_gaussians),
    ("scale length == 3K",                  normalization_scale.size == 3 * loaded_run.n_gaussians),
    ("all scales strictly positive",        bool((normalization_scale > 0).all())),
    ("denorm output shape preserved",       denormalized_zero.shape == (1, 3 * loaded_run.n_gaussians, 4, 4)),
    ("clamped amplitudes non-negative",     bool((amplitude_channels >= 0.0).all())),
    ("clamped means within elevation axis", bool((mean_channels >= elevation_minimum - 1e-3).all() and (mean_channels <= elevation_maximum + 1e-3).all())),
    ("clamped sigmas positive",             bool((sigma_channels > 0.0).all())),
    ("denorm output finite",                bool(np.isfinite(denormalized_zero).all())),
]
for stage_3_description, stage_3_condition in stage_3_checks:
    print(f"[{'PASS' if stage_3_condition else 'FAIL'}] {stage_3_description}")

### Common Mistakes — Stage 3: Model Wrapper

| Symptom | Likely Cause | How to Diagnose |
|---------|--------------|-----------------|
| All metrics enormous, params off by orders of magnitude | Training normalization not inverted at inference (raw normalized output treated as physical) | Compare denormalized-zero against the `loc` vector; they should roughly match |
| De-normalization applied in the wrong direction | `(p - loc)/scale` used instead of `p*scale + loc` | Feed a zero tensor: correct inverse yields `loc`, the wrong direction yields `-loc/scale` |
| Predicted means $\mu$ fall outside the scene | Elevation axis passed to the clamp differs from the training `x_axis` | Confirm `mean_channels` lie inside `[x_axis.min(), x_axis.max()]` |
| Negative amplitudes or zero sigmas leak through | `clamp_gaussian_params` skipped (wrapper bypassed and raw model called directly) | Ensure predictions come through `ModelWrapper`, not the bare `nn.Module` |
| `amp_max` clipping flattens tall peaks | `GaussianConfig.amp_max` does not match the training data scale | Inspect amplitude histogram in Stage 5 for a hard ceiling at `amp_max` |

**Passing to Stage 4:** `model_wrapper` — the callable that, given a patch image tensor, returns denormalized clamped Gaussian parameters; `normalization_location` / `normalization_scale` — the per-channel statistics reused for GT de-normalization.

---
## Stage 4: Per-patch GPU Forward Pass

**Callable:** `Predictor._forward_pass()`
**Input:** `run.loader`, yielding `(image_patch, gt_parameters)` batches; image patches have shape `(B, in_channels, ph, pw)`.
**Output:** three aligned lists over batches — `all_indices` (flat patch indices), `all_pred_params` (denormalized predicted parameters, `ndarray (B, 3K, ph, pw)`), and `all_gt_params` (denormalized ground-truth parameters, `ndarray (B, 3K, ph, pw)`).

For each patch the wrapped model maps the input image to $3K$ parameter channels; the GT parameters are sliced to the first $3K$ channels and de-normalized with the same statistics. No spatial transformation occurs here — patches retain their `(ph, pw)` footprint.

> **What you should see:** the concatenation of `all_indices` is exactly `range(grid.number_of_patches)` with no gaps or repeats; every predicted-params block has `3 * run.n_gaussians` channels and patch spatial size `run.grid.patch_size`; predicted and GT blocks have identical shapes; all values are finite.

In [ ]:
predictor = Predictor(
    run         = loaded_run,
    logger      = inference_logger,
    window_kind = inference_configuration.stitch_window,
    cube_dtype  = inference_configuration.cube_dtype,
    save_cubes  = inference_configuration.save_cubes,
    meta        = inference_metadata,
    cpu_workers = inference_configuration.cpu_workers,
)

forward_pass_indices, forward_pass_pred_params, forward_pass_gt_params = predictor._forward_pass()

In [ ]:
concatenated_indices = np.concatenate([np.asarray(batch_indices) for batch_indices in forward_pass_indices])
first_pred_block     = forward_pass_pred_params[0]
first_gt_block       = forward_pass_gt_params[0]

print("number of batches        :", len(forward_pass_pred_params))
print("total patches covered    :", concatenated_indices.size)
print("grid number_of_patches   :", loaded_run.grid.number_of_patches)
print()
print("first pred block type    :", type(first_pred_block).__name__)
print("first pred block shape   :", np.asarray(first_pred_block).shape)
print("first pred block dtype   :", np.asarray(first_pred_block).dtype)
print("first gt   block shape   :", np.asarray(first_gt_block).shape)
print()
predicted_parameter_stack = np.asarray(first_pred_block)
print("pred params min / max    :", float(predicted_parameter_stack.min()), float(predicted_parameter_stack.max()))
print("pred params mean / std   :", float(predicted_parameter_stack.mean()), float(predicted_parameter_stack.std()))
print("pred params finite       :", bool(np.isfinite(predicted_parameter_stack).all()))

In [ ]:
first_batch_pred = np.asarray(forward_pass_pred_params[0])
first_batch_gt   = np.asarray(forward_pass_gt_params[0])

parameter_kind_labels = ["amplitude a", "mean mu [m]", "std-dev sigma"]
figure, axes = plt.subplots(1, 3, figsize=(15, 4))

for kind_index, (axis, kind_label) in enumerate(zip(axes, parameter_kind_labels)):
    predicted_values = first_batch_pred[:, kind_index::3].reshape(-1)
    ground_values    = first_batch_gt[:, kind_index::3].reshape(-1)
    predicted_values = predicted_values[np.isfinite(predicted_values)]
    ground_values    = ground_values[np.isfinite(ground_values)]
    lower_bound      = float(np.percentile(np.concatenate([predicted_values, ground_values]), 0.5))
    upper_bound      = float(np.percentile(np.concatenate([predicted_values, ground_values]), 99.5))
    axis.hist(ground_values,    bins=60, range=(lower_bound, upper_bound), density=True, color="C0", alpha=0.55, label="GT")
    axis.hist(predicted_values, bins=60, range=(lower_bound, upper_bound), density=True, color="C3", alpha=0.55, label="Pred")
    axis.set_xlabel(kind_label)
    axis.set_ylabel("density")
    axis.set_title(kind_label)
    axis.legend()
    axis.grid(True, linewidth=0.3, alpha=0.4)

figure.suptitle("Stage 4 — Forward Pass: first-batch denormalized parameter distributions (GT vs Pred)")
figure.tight_layout(rect=(0, 0, 1, 0.95))
figure.savefig(figure_directory / "stage_4_forward_pass.pdf")
plt.show()

In [ ]:
expected_patch_channels = 3 * loaded_run.n_gaussians
expected_patch_height, expected_patch_width = loaded_run.grid.patch_size
all_blocks_correct_shape = all(
    np.asarray(predicted_block).shape[1:] == (expected_patch_channels, expected_patch_height, expected_patch_width)
    for predicted_block in forward_pass_pred_params
)
pred_gt_shapes_match = all(
    np.asarray(predicted_block).shape == np.asarray(ground_block).shape
    for predicted_block, ground_block in zip(forward_pass_pred_params, forward_pass_gt_params)
)

stage_4_checks = [
    ("indices cover every patch once",      np.array_equal(np.sort(concatenated_indices), np.arange(loaded_run.grid.number_of_patches))),
    ("total patches == grid patches",       concatenated_indices.size == loaded_run.grid.number_of_patches),
    ("pred blocks have 3K channels",        all_blocks_correct_shape),
    ("pred and gt shapes match",            pred_gt_shapes_match),
    ("first pred block finite",             bool(np.isfinite(first_batch_pred).all())),
    ("first gt block finite",               bool(np.isfinite(first_batch_gt).all())),
    ("batch count matches loader length",   len(forward_pass_pred_params) == len(loaded_run.loader)),
]
for stage_4_description, stage_4_condition in stage_4_checks:
    print(f"[{'PASS' if stage_4_condition else 'FAIL'}] {stage_4_description}")

### Common Mistakes — Stage 4: Forward Pass

| Symptom | Likely Cause | How to Diagnose |
|---------|--------------|-----------------|
| Patch indices skip or repeat | `DataLoader` reordered or `drop_last=True` dropping the final partial batch | Verify `np.sort(concatenated_indices)` equals `arange(number_of_patches)` |
| GT params on a different scale than predictions | GT not de-normalized with the same `denormalize_output` | Both histograms in the figure should overlap in support |
| Activations / outputs vary run to run | Model left in train mode (dropout / BatchNorm noise) | Re-check `model.eval()` from Stage 2; outputs must be deterministic |
| Shapes `(B, 3K+1, ...)` | Noise-head channel not stripped before reshaping into triples | Confirm channel count is exactly `3 * n_gaussians` |
| Out-of-memory during forward pass | Batch size too large for the device | Lower `batch_size` in the config; it propagates to the loader |
| Edge patches contain padded rows / columns | Reflective padding applied at patch extraction but not cropped at stitch time | Padding is removed in Stage 6 `finalize_cube`; verify the crop offsets there |

**Passing to Stage 5:** `forward_pass_indices`, `forward_pass_pred_params`, `forward_pass_gt_params` — aligned per-batch lists of flat patch indices and denormalized predicted / ground-truth Gaussian parameters.

---
## Stage 5: Curve Reconstruction and Per-patch Metrics

**Callable:** `Predictor._compute_metrics(...)` (CPU pool over `Predictor._cpu_worker`, which calls `GaussianReconstructor.reconstruct_batch`).
**Input:** the per-batch lists of denormalized predicted / GT parameters from Stage 4.
**Output:** one tuple per batch: `(pred_curves, gt_curves, pred_gauss_flat, gt_gauss_flat, {mse, mae, r2, cos}, peak_err)`, where curves have shape `(B, N_elev, ph, pw)` and each metric map has shape `(B, ph, pw)`.

Per pixel, the predicted elevation profile is the Gaussian superposition
$$ \hat y(x_n) = \sum_{k=1}^{K} \max(a_k, 0)\, \exp\!\left( -\frac{(x_n - \mu_k)^2}{2\sigma_k^2 + \epsilon} \right). $$
GT slots are first $\mu$-sorted (placeholder slots, $a < 10^{-3}$, pushed to the end). The per-pixel metrics over the $N$ elevation samples are
$$ \text{MSE} = \tfrac1N\sum_n (\hat y_n - y_n)^2,\quad R^2 = 1 - \frac{\sum_n(\hat y_n - y_n)^2}{\sum_n(y_n - \bar y)^2 + \epsilon},\quad \cos = \frac{\sum_n \hat y_n y_n}{\lVert\hat y\rVert\,\lVert y\rVert}. $$

> **What you should see:** reconstructed curves are non-negative (amplitudes clamped at 0) with shape `(B, N_elev, ph, pw)`; per-pixel $R^2 \le 1$ and cosine in $[-1, 1]$; MSE $\ge 0$; the peak-error map holds non-negative integer index distances.

In [ ]:
stage_5_cpu_results = predictor._compute_metrics(forward_pass_pred_params, forward_pass_gt_params)

(first_pred_curves, first_gt_curves, first_pred_gauss_flat, first_gt_gauss_flat, first_metric_maps, first_peak_error) = stage_5_cpu_results[0]

In [ ]:
print("number of result tuples  :", len(stage_5_cpu_results))
print("pred_curves shape         :", first_pred_curves.shape, "(B, N_elev, ph, pw)")
print("gt_curves   shape         :", first_gt_curves.shape)
print("pred_gauss_flat shape     :", first_pred_gauss_flat.shape)
print("metric map keys           :", list(first_metric_maps.keys()))
print("per-pixel mse map shape   :", first_metric_maps['mse'].shape)
print("peak error map shape      :", first_peak_error.shape)
print()
print("pred_curves min / max     :", float(first_pred_curves.min()), float(first_pred_curves.max()))
print("gt_curves   min / max     :", float(first_gt_curves.min()), float(first_gt_curves.max()))
print("per-pixel R2  min/median/max :", float(np.nanmin(first_metric_maps['r2'])), float(np.nanmedian(first_metric_maps['r2'])), float(np.nanmax(first_metric_maps['r2'])))
print("per-pixel cos min/median/max :", float(np.nanmin(first_metric_maps['cos'])), float(np.nanmedian(first_metric_maps['cos'])), float(np.nanmax(first_metric_maps['cos'])))
print("per-pixel mse min/median/max :", float(np.nanmin(first_metric_maps['mse'])), float(np.nanmedian(first_metric_maps['mse'])), float(np.nanmax(first_metric_maps['mse'])))

In [ ]:
center_row    = first_pred_curves.shape[2] // 2
center_column = first_pred_curves.shape[3] // 2
elevation_axis_sorted_index = np.argsort(loaded_run.x_axis)
elevation_axis_values       = loaded_run.x_axis[elevation_axis_sorted_index]

predicted_profile = first_pred_curves[0, :, center_row, center_column][elevation_axis_sorted_index]
ground_profile    = first_gt_curves[0, :, center_row, center_column][elevation_axis_sorted_index]

figure, (axis_left, axis_right) = plt.subplots(1, 2, figsize=(13, 4.2))

axis_left.plot(elevation_axis_values, ground_profile,    color="black", linewidth=1.4, label="GT (Gaussian)")
axis_left.plot(elevation_axis_values, predicted_profile, color="C3", linewidth=1.2, linestyle="--", label="Prediction")
axis_left.fill_between(elevation_axis_values, predicted_profile - ground_profile, 0.0, color="C0", alpha=0.15, linewidth=0, label="residual")
axis_left.set_xlabel("elevation [m]")
axis_left.set_ylabel("intensity")
axis_left.set_title(f"Reconstructed elevation profile (patch centre pixel {center_row}, {center_column})")
axis_left.legend()
axis_left.grid(True, linewidth=0.3, alpha=0.4)

per_pixel_r2 = first_metric_maps['r2'].reshape(-1)
per_pixel_r2 = per_pixel_r2[np.isfinite(per_pixel_r2)]
axis_right.hist(per_pixel_r2, bins=60, range=(max(-1.0, float(np.percentile(per_pixel_r2, 1))), 1.0), color="C2", edgecolor="white", linewidth=0.3)
axis_right.axvline(float(np.median(per_pixel_r2)), color="black", linestyle="--", linewidth=1.0, label=f"median R2 = {np.median(per_pixel_r2):.3f}")
axis_right.set_xlabel("per-pixel R2 (first batch)")
axis_right.set_ylabel("count")
axis_right.set_title("Per-pixel reconstruction R2")
axis_right.legend()
axis_right.grid(True, linewidth=0.3, alpha=0.4)

figure.suptitle("Stage 5 — Reconstruction: profile fit and per-pixel R2")
figure.tight_layout(rect=(0, 0, 1, 0.95))
figure.savefig(figure_directory / "stage_5_reconstruction.pdf")
plt.show()

In [ ]:
finite_cosine = first_metric_maps['cos'][np.isfinite(first_metric_maps['cos'])]
finite_r2     = first_metric_maps['r2'][np.isfinite(first_metric_maps['r2'])]

stage_5_checks = [
    ("result tuple count == batch count",   len(stage_5_cpu_results) == len(forward_pass_pred_params)),
    ("pred_curves elev dim == x_axis_length", first_pred_curves.shape[1] == loaded_run.x_axis_length),
    ("pred_curves spatial == patch_size",    first_pred_curves.shape[2:] == loaded_run.grid.patch_size),
    ("reconstructed curves non-negative",    bool((first_pred_curves >= -1e-6).all())),
    ("per-pixel MSE non-negative",           bool((first_metric_maps['mse'] >= 0.0).all())),
    ("per-pixel R2 <= 1",                    bool((finite_r2 <= 1.0 + 1e-6).all())),
    ("per-pixel cosine in [-1, 1]",          bool((finite_cosine >= -1.0 - 1e-6).all() and (finite_cosine <= 1.0 + 1e-6).all())),
    ("peak error map non-negative",          bool((first_peak_error >= 0.0).all())),
    ("curves finite",                        bool(np.isfinite(first_pred_curves).all() and np.isfinite(first_gt_curves).all())),
]
for stage_5_description, stage_5_condition in stage_5_checks:
    print(f"[{'PASS' if stage_5_condition else 'FAIL'}] {stage_5_description}")

### Common Mistakes — Stage 5: Reconstruction & Per-patch Metrics

| Symptom | Likely Cause | How to Diagnose |
|---------|--------------|-----------------|
| Reconstructed profile is the mirror image of GT | Elevation axis `x` not aligned / sorted between pred and GT reconstruction | Plot pred vs GT against the *sorted* `x_axis`; peaks should coincide |
| R2 wildly negative for many pixels | Pred and GT Gaussian slots not consistently ordered (permutation mismatch) | GT is $\mu$-sorted in `_cpu_worker`; verify pred is sorted the same way |
| Curves clipped to zero everywhere | All amplitudes clamped to 0 (denormalization sign error upstream) | Inspect amplitude histogram from Stage 4 |
| Cosine similarity is `nan` | Zero-energy GT or pred profile (norm = 0) divided without the `1e-8` guard | The guard exists; flat-zero pixels (background) can still be near-degenerate |
| Peak-error in samples not metres | Peak distance left as an index difference, not multiplied by `x_step` | The index map is converted to metres only in Stage 8 metrics |
| Reconstruction extremely slow | Too few `cpu_workers`, or per-pixel Python loop instead of vectorized batch | `reconstruct_batch` is vectorized; check `cpu_workers` in the config |

**Passing to Stage 6:** `stage_5_cpu_results` — per-batch tuples of reconstructed curves, flattened Gaussian parameters, and per-pixel metric maps, aligned with `forward_pass_indices`.

---
## Stage 6: Stitching — windowed overlap-add into the full-scene cube

**Callable:** `Predictor._stitch_results(all_indices, cpu_results)` (uses `CubeStitcher` and `CubeStitcher.make_patch_window`), finalized by `Predictor._finalize_results(...)`.
**Input:** the per-batch curves / params / metric maps from Stage 5, plus the patch `GridInfo`.
**Output:** a `Result` dataclass holding the full-scene cubes `pred_curves`, `gt_curves`, `params_pred`, `params_gt` of shape `(N_elev, azimuth, range)` / `(3K, azimuth, range)`, the per-pixel metric maps `pixel_mse / pixel_mae / pixel_r2 / pixel_cosine / pixel_peak_err_idx` of shape `(azimuth, range)`, the `cube_directory`, and the `azimuth_offset` / `range_offset`.

Each patch $p$ contributes through a 2-D window $w$ (Hann by default). The blended value at a scene location is the weighted average over all patches covering it:
$$ \text{cube}(c, i, j) = \frac{\sum_{p} w_p(i, j)\, \text{patch}_p(c, i, j)}{\sum_{p} w_p(i, j)}. $$
The Hann window is $w(n) = 0.5 - 0.5\cos\!\big(2\pi (n + 0.5)/L\big)$ per axis (outer product across the two patch axes), clipped to a $10^{-3}$ floor so no covered pixel has zero weight. Reflective padding added at extraction is cropped away in `finalize_cube`.

> **What you should see:** cube spatial size equals `run.grid.spatial_size` `(azimuth_size, range_size)` exactly (padding removed); `pred_curves` has `x_axis_length` elevation bins; the per-pixel maps have no `inf` and finite means; `azimuth_offset` / `range_offset` equal the split region start; the weight map of the stitcher is strictly positive over every covered pixel (no seams).

In [ ]:
stage_6_result = predictor._stitch_results(forward_pass_indices, stage_5_cpu_results)
inference_result = stage_6_result

In [ ]:
print("pred_curves cube shape  :", inference_result.pred_curves.shape, "(N_elev, azimuth, range)")
print("gt_curves   cube shape  :", inference_result.gt_curves.shape)
print("params_pred cube shape  :", inference_result.params_pred.shape, "(3K, azimuth, range)")
print("params_gt   cube shape  :", inference_result.params_gt.shape)
print()
print("pixel_mse map shape     :", inference_result.pixel_mse.shape)
print("pixel_r2  map shape     :", inference_result.pixel_r2.shape)
print("pixel_peak idx map dtype:", inference_result.pixel_peak_err_idx.dtype)
print()
print("azimuth_offset          :", inference_result.azimuth_offset)
print("range_offset            :", inference_result.range_offset)
print("cube_directory          :", inference_result.cube_directory)
print()
print("pred_curves min / max   :", float(inference_result.pred_curves.min()), float(inference_result.pred_curves.max()))
print("pixel_mse mean          :", float(inference_result.pixel_mse.mean()))
print("pixel_r2  mean          :", float(np.nanmean(inference_result.pixel_r2)))
print("pixel maps contain inf  :", bool(np.isinf(inference_result.pixel_mse).any() or np.isinf(inference_result.pixel_r2).any()))

In [ ]:
patch_window      = CubeStitcher.make_patch_window(loaded_run.grid.patch_size, kind=inference_configuration.stitch_window)
diagnostic_stitcher = CubeStitcher(grid=loaded_run.grid, n_channels=1, window_kind=inference_configuration.stitch_window, dtype="float32")
for diagnostic_patch_index in range(loaded_run.grid.number_of_patches):
    diagnostic_stitcher.add_patch(diagnostic_patch_index, np.ones((1, *loaded_run.grid.patch_size), dtype=np.float32))
scene_weight_padded = diagnostic_stitcher._weight
padding_top, padding_left = loaded_run.grid.pad_top, loaded_run.grid.pad_left
scene_height, scene_width = loaded_run.grid.spatial_size
scene_weight = scene_weight_padded[padding_top:padding_top + scene_height, padding_left:padding_left + scene_width]

elevation_average_pred = inference_result.pred_curves.mean(axis=0)
elevation_average_gt   = inference_result.gt_curves.mean(axis=0)

figure, axes = plt.subplots(2, 2, figsize=(13, 9))

window_image = axes[0, 0].imshow(patch_window, cmap="viridis", aspect="auto", interpolation="nearest")
axes[0, 0].set_title(f"Patch blend window ({inference_configuration.stitch_window})")
axes[0, 0].set_xlabel("patch range index")
axes[0, 0].set_ylabel("patch azimuth index")
figure.colorbar(window_image, ax=axes[0, 0], fraction=0.046, pad=0.02).set_label("window weight")

weight_image = axes[0, 1].imshow(scene_weight, cmap="magma", aspect="auto", interpolation="nearest", extent=[inference_result.range_offset, inference_result.range_offset + scene_width, inference_result.azimuth_offset + scene_height, inference_result.azimuth_offset])
axes[0, 1].set_title("Accumulated overlap weight (seam check)")
axes[0, 1].set_xlabel("range index")
axes[0, 1].set_ylabel("azimuth index")
figure.colorbar(weight_image, ax=axes[0, 1], fraction=0.046, pad=0.02).set_label("sum of window weights")

gt_image = axes[1, 0].imshow(elevation_average_gt, cmap=inference_configuration.cmap_intensity, aspect="auto", interpolation="nearest", extent=[inference_result.range_offset, inference_result.range_offset + scene_width, inference_result.azimuth_offset + scene_height, inference_result.azimuth_offset])
axes[1, 0].set_title("Stitched GT scene (elevation-averaged intensity)")
axes[1, 0].set_xlabel("range index")
axes[1, 0].set_ylabel("azimuth index")
figure.colorbar(gt_image, ax=axes[1, 0], fraction=0.046, pad=0.02).set_label("mean intensity")

pred_image = axes[1, 1].imshow(elevation_average_pred, cmap=inference_configuration.cmap_intensity, aspect="auto", interpolation="nearest", extent=[inference_result.range_offset, inference_result.range_offset + scene_width, inference_result.azimuth_offset + scene_height, inference_result.azimuth_offset])
axes[1, 1].set_title("Stitched prediction scene (elevation-averaged intensity)")
axes[1, 1].set_xlabel("range index")
axes[1, 1].set_ylabel("azimuth index")
figure.colorbar(pred_image, ax=axes[1, 1], fraction=0.046, pad=0.02).set_label("mean intensity")

figure.suptitle("Stage 6 — Stitching: blend window, overlap weights, full-scene reconstruction")
figure.tight_layout(rect=(0, 0, 1, 0.96))
figure.savefig(figure_directory / "stage_6_stitching.pdf")
plt.show()

In [ ]:
expected_scene_height, expected_scene_width = loaded_run.grid.spatial_size

stage_6_checks = [
    ("isinstance Result",                    isinstance(inference_result, Result)),
    ("pred_curves spatial == scene size",    inference_result.pred_curves.shape[1:] == (expected_scene_height, expected_scene_width)),
    ("pred_curves elev == x_axis_length",    inference_result.pred_curves.shape[0] == loaded_run.x_axis_length),
    ("params_pred channels == 3K",           inference_result.params_pred.shape[0] == 3 * loaded_run.n_gaussians),
    ("pred and gt cubes same shape",         inference_result.pred_curves.shape == inference_result.gt_curves.shape),
    ("pixel maps == scene size",             inference_result.pixel_mse.shape == (expected_scene_height, expected_scene_width)),
    ("overlap weight strictly positive",     bool((scene_weight > 0.0).all())),
    ("no inf in pixel_mse",                  bool(np.isfinite(inference_result.pixel_mse).all())),
    ("azimuth_offset == split start",        inference_result.azimuth_offset == loaded_run.split_region.azimuth_start),
    ("range_offset == split start",          inference_result.range_offset == loaded_run.split_region.range_start),
    ("pred_curves non-negative",             bool((inference_result.pred_curves >= -1e-6).all())),
]
for stage_6_description, stage_6_condition in stage_6_checks:
    print(f"[{'PASS' if stage_6_condition else 'FAIL'}] {stage_6_description}")

### Common Mistakes — Stage 6: Stitching

| Symptom | Likely Cause | How to Diagnose |
|---------|--------------|-----------------|
| Visible grid seams in the stitched scene | Uniform (boxcar) window instead of Hann, or stride equal to patch size (no overlap) | View the overlap-weight map; Hann blending should be smooth, not blocky |
| Bright/dark stripes at patch boundaries | Window not normalized by accumulated weight (overlap-add without dividing by the weight sum) | `finalize_cube` divides by `weight_safe`; confirm the weight map has no zeros |
| Stitched scene larger than the scene | Reflective padding not cropped after blending | Cube spatial size must equal `grid.spatial_size`, not `grid.padded_size` |
| Whole scene offset by a fixed shift | `azimuth_offset` / `range_offset` not taken from the split region | Offsets must equal `split_region.azimuth_start` / `range_start` |
| Patch placed at the wrong location | `divmod(idx, n_h)` vs `n_v` mix-up mapping flat index to grid cell | Cross-check `_stitch_results` placement against `CubeStitcher.add_patch` |
| `NaN` in the param cubes for background pixels | Placeholder ($a<10^{-7}$) `mu`/`sigma` deliberately set to NaN in `finalize_results` | Expected for inactive slots; mask before computing statistics |

**Passing to Stage 7:** `inference_result` — the full-scene `Result` (cubes + per-pixel maps + offsets) that every remaining stage consumes.

---
## Stage 7: Slice Index Selection

**Callable:** `InferencePipeline._compute_slice_indices(config, n_elev, n_az, n_rg)`
**Input:** the full-scene cube dimensions and the configured slice counts.
**Output:** a dict of index arrays — `slice_elev_idx`, `slice_range_idx`, `slice_az_idx` (evenly spaced representative slices), and `all_elev_idx`, `all_range_idx`, `all_az_idx` (every index along each axis).

Representative slices are placed on a linear grid spanning the inner 10%-90% of each axis:
$$ \text{idx}_i = \operatorname{round}\!\left(0.1\,n + \frac{0.8\,n\,(i)}{m-1}\right),\quad i = 0,\dots,m-1, $$
with $m = \min(\text{n\_slices}, n)$. The `all_*` arrays drive the dense per-slice SSIM and per-elevation-bin metric curves in Stage 8.

> **What you should see:** each `slice_*` array has length `min(config.n_*_slices, axis_length)`, holds strictly increasing integer indices inside the axis bounds (and inside the inner 10%-90% band); each `all_*` array equals `arange(axis_length)`.

In [ ]:
stage_7_n_elev, stage_7_n_az, stage_7_n_rg = inference_result.pred_curves.shape

slice_indices = InferencePipeline(inference_configuration)._compute_slice_indices(
    inference_configuration, stage_7_n_elev, stage_7_n_az, stage_7_n_rg,
)

In [ ]:
print("cube dimensions (elev, az, rg) :", (stage_7_n_elev, stage_7_n_az, stage_7_n_rg))
print()
for slice_key in ("slice_elev_idx", "slice_range_idx", "slice_az_idx"):
    selected_indices = slice_indices[slice_key]
    print(f"{slice_key:<16}: length={selected_indices.size:<3} indices={selected_indices.tolist()}")
print()
for all_key in ("all_elev_idx", "all_range_idx", "all_az_idx"):
    print(f"{all_key:<16}: length={slice_indices[all_key].size}")

In [ ]:
axis_specifications = [
    ("slice_elev_idx",  stage_7_n_elev, "elevation"),
    ("slice_az_idx",    stage_7_n_az,   "azimuth"),
    ("slice_range_idx", stage_7_n_rg,   "range"),
]

figure, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for axis, (slice_key, axis_length, axis_label) in zip(axes, axis_specifications):
    selected_indices = slice_indices[slice_key]
    axis.hlines(0, 0, axis_length - 1, color="#bbbbbb", linewidth=2)
    axis.scatter(selected_indices, np.zeros_like(selected_indices), color="C3", s=60, zorder=3, label="selected slice")
    axis.axvspan(0.1 * axis_length, 0.9 * axis_length, color="C0", alpha=0.12, label="10%-90% band")
    axis.set_xlim(-0.02 * axis_length, 1.02 * axis_length)
    axis.set_yticks([])
    axis.set_xlabel(f"{axis_label} index")
    axis.set_title(f"{axis_label} slices ({selected_indices.size} of {axis_length})")
    axis.legend(loc="upper center", fontsize=8)

figure.suptitle("Stage 7 — Slice Index Selection: representative slice placement")
figure.tight_layout(rect=(0, 0, 1, 0.94))
figure.savefig(figure_directory / "stage_7_slice_indices.pdf")
plt.show()

In [ ]:
expected_elev_slice_count  = min(inference_configuration.n_elevation_slices, stage_7_n_elev)
expected_range_slice_count = min(inference_configuration.n_range_slices,     stage_7_n_rg)
expected_az_slice_count    = min(inference_configuration.n_azimuth_slices,   stage_7_n_az)

def _is_strictly_increasing(index_array):
    return bool(np.all(np.diff(index_array) > 0)) if index_array.size > 1 else True

stage_7_checks = [
    ("elev slice count matches config",     slice_indices['slice_elev_idx'].size == expected_elev_slice_count),
    ("range slice count matches config",    slice_indices['slice_range_idx'].size == expected_range_slice_count),
    ("azimuth slice count matches config",  slice_indices['slice_az_idx'].size == expected_az_slice_count),
    ("elev slices strictly increasing",     _is_strictly_increasing(slice_indices['slice_elev_idx'])),
    ("range slices in bounds",              bool((slice_indices['slice_range_idx'] >= 0).all() and (slice_indices['slice_range_idx'] < stage_7_n_rg).all())),
    ("azimuth slices in bounds",            bool((slice_indices['slice_az_idx'] >= 0).all() and (slice_indices['slice_az_idx'] < stage_7_n_az).all())),
    ("all_elev_idx == arange(n_elev)",      np.array_equal(slice_indices['all_elev_idx'], np.arange(stage_7_n_elev))),
    ("all_range_idx == arange(n_rg)",       np.array_equal(slice_indices['all_range_idx'], np.arange(stage_7_n_rg))),
    ("all_az_idx == arange(n_az)",          np.array_equal(slice_indices['all_az_idx'], np.arange(stage_7_n_az))),
]
for stage_7_description, stage_7_condition in stage_7_checks:
    print(f"[{'PASS' if stage_7_condition else 'FAIL'}] {stage_7_description}")

### Common Mistakes — Stage 7: Slice Index Selection

| Symptom | Likely Cause | How to Diagnose |
|---------|--------------|-----------------|
| Slice figures all show the same plane | Slice count exceeds axis length and collapses to one index | Check `slice_*` length equals `min(config, axis_length)` |
| Selected slices land on padded / edge rows | Band not restricted to the inner 10%-90% | Slices should fall inside the shaded band in the figure |
| `IndexError` slicing the cube downstream | Slice index axis swapped (azimuth count used for range, etc.) | Confirm cube shape `(n_elev, n_az, n_rg)` order matches the index keys |
| Duplicate slice indices | `round` collapses neighbours when axis is short | Strictly-increasing check flags this |
| Dense SSIM curves empty | `all_*_idx` not passed to `Metrics.compute` | The `all_*` arrays must reach Stage 8 |

**Passing to Stage 8:** `slice_indices` — dict of representative and dense index arrays for each cube axis, consumed by both metric computation and figure rendering.

---
## Stage 8: Global Metrics Computation

**Callable:** `Metrics(result, x_axis, n_gaussians).compute(elev_indices, range_indices, az_indices)` then `Metrics.write_json(...)`.
**Input:** the full-scene `Result`, the elevation `x_axis`, and the dense `all_*` index arrays.
**Output:** a flat `Dict[str, float]` of curve-level, per-pixel, SSIM, per-elevation-bin, and Gaussian-parameter / slot-organisation metrics, written to `metrics.json`.

Curve-level aggregates over the whole cube: $\text{RMSE} = \sqrt{\text{MSE}}$, $R^2_{\text{overall}} = 1 - \frac{\sum(\hat y - y)^2}{\sum(y - \bar y)^2 + \epsilon}$, and $\text{PSNR} = 10\log_{10}\!\frac{(\max y - \min y)^2}{\text{MSE}}$. SSIM is computed per slice along each axis and averaged. Peak-location error is converted to physical units via the elevation step $\Delta x$.

> **What you should see:** `curve_rmse_gt == sqrt(curve_mse_gt)`; `overall_r2_gt <= 1`; SSIM means in $[-1, 1]$ (typically high, near 1, for a well-fit model); `n_pixels` equals the scene azimuth times range; `pixel_r2_gt_mean` is finite; all reported values are JSON-serializable scalars.

In [ ]:
elevation_axis_for_metrics = np.asarray(loaded_run.x_axis, dtype=np.float64)

global_metrics = Metrics(inference_result, elevation_axis_for_metrics, loaded_run.n_gaussians).compute(
    elev_indices  = slice_indices['all_elev_idx'],
    range_indices = slice_indices['all_range_idx'],
    az_indices    = slice_indices['all_az_idx'],
)

written_metrics_path = Metrics.write_json(global_metrics, inference_metadata.metrics_path)

In [ ]:
print("metrics written to       :", written_metrics_path)
print("total metric keys        :", len(global_metrics))
print()
for metric_key in ("curve_mse_gt", "curve_mae_gt", "curve_rmse_gt", "overall_r2_gt", "psnr_db_gt"):
    print(f"  {metric_key:<16}: {global_metrics.get(metric_key)}")
print()
for metric_key in ("pixel_mse_gt_mean", "pixel_mse_gt_median", "pixel_r2_gt_mean", "pixel_r2_gt_median", "pixel_cosine_gt_mean"):
    print(f"  {metric_key:<20}: {global_metrics.get(metric_key)}")
print()
for metric_key in ("ssim_gt_elev_mean", "ssim_gt_range_mean", "ssim_gt_azimuth_mean"):
    print(f"  {metric_key:<20}: {global_metrics.get(metric_key)}")
print()
for metric_key in ("n_pixels", "n_elevation", "x_axis_step", "pixel_peak_err_units_mean_gt", "mu_ordering_rate"):
    print(f"  {metric_key:<28}: {global_metrics.get(metric_key)}")

In [ ]:
elevation_metric_specifications = [
    ("elev_mae_gt",  "MAE"),
    ("elev_rmse_gt", "RMSE"),
    ("elev_r2_gt",   "R2"),
    ("elev_ce_gt",   "cross-entropy"),
]

figure, axes = plt.subplots(2, 2, figsize=(13, 7))
for axis, (metric_prefix, metric_label) in zip(axes.ravel(), elevation_metric_specifications):
    per_bin_values = np.array([global_metrics.get(f"{metric_prefix}_{bin_index}", np.nan) for bin_index in range(stage_7_n_elev)], dtype=np.float64)
    axis.plot(loaded_run.x_axis, per_bin_values, color="C0", linewidth=0.9)
    axis.axhline(float(np.nanmean(per_bin_values)), color="black", linestyle="--", linewidth=1.0, label=f"mean = {np.nanmean(per_bin_values):.4g}")
    axis.set_xlabel("elevation [m]")
    axis.set_ylabel(metric_label)
    axis.set_title(f"{metric_label} per elevation bin")
    axis.legend()
    axis.grid(True, linewidth=0.3, alpha=0.4)

figure.suptitle("Stage 8 — Global Metrics: per-elevation-bin error curves")
figure.tight_layout(rect=(0, 0, 1, 0.96))
figure.savefig(figure_directory / "stage_8_global_metrics.pdf")
plt.show()

In [ ]:
expected_pixel_count = inference_result.pixel_mse.size
ssim_means = [global_metrics.get("ssim_gt_elev_mean"), global_metrics.get("ssim_gt_range_mean"), global_metrics.get("ssim_gt_azimuth_mean")]
ssim_means_finite = [value for value in ssim_means if value is not None and np.isfinite(value)]

stage_8_checks = [
    ("metrics.json written",                written_metrics_path.exists()),
    ("rmse == sqrt(mse)",                   abs(global_metrics['curve_rmse_gt'] - np.sqrt(global_metrics['curve_mse_gt'])) < 1e-6),
    ("overall R2 <= 1",                     global_metrics['overall_r2_gt'] <= 1.0 + 1e-6),
    ("curve MSE non-negative",              global_metrics['curve_mse_gt'] >= 0.0),
    ("n_pixels == scene az * rg",           global_metrics['n_pixels'] == expected_pixel_count),
    ("n_elevation == x_axis_length",        global_metrics['n_elevation'] == loaded_run.x_axis_length),
    ("pixel_r2 mean finite",                np.isfinite(global_metrics['pixel_r2_gt_mean'])),
    ("all SSIM means within [-1, 1]",       all(-1.0 - 1e-6 <= value <= 1.0 + 1e-6 for value in ssim_means_finite)),
    ("x_axis_step matches median spacing",  abs(global_metrics['x_axis_step'] - float(loaded_run.x_axis[1] - loaded_run.x_axis[0])) < 1e-6),
    ("all metric values are scalars",       all(np.isscalar(value) or isinstance(value, (int, float)) for value in global_metrics.values())),
]
for stage_8_description, stage_8_condition in stage_8_checks:
    print(f"[{'PASS' if stage_8_condition else 'FAIL'}] {stage_8_description}")

### Common Mistakes — Stage 8: Global Metrics

| Symptom | Likely Cause | How to Diagnose |
|---------|--------------|-----------------|
| Metrics computed on normalized, not physical, curves | Metrics fed pre-denormalization parameters | Curves come from Stage 6 denormalized cubes; magnitudes should match GT intensity |
| `overall_r2_gt` strongly negative | Orientation / coordinate flip between pred and GT cube | Compare a tomogram slice visually in Stage 9 |
| Peak error reported in samples, not metres | `x_step` multiplication omitted | `pixel_peak_err_units_*` must scale by `x_axis_step` |
| SSIM returns `nan` for flat slices | Constant slice has zero data range, SSIM undefined | Mean over slices uses only finite values; check individual slice SSIMs |
| PSNR is `inf` | MSE is exactly zero (pred identical to GT) or degenerate slice | Inspect `curve_mse_gt`; `inf` PSNR implies a trivial match |
| `metrics.json` write fails on numpy types | Non-serializable numpy scalars | `write_json` uses `default=str`; verify the file parses back |

**Passing to Stage 9:** `global_metrics` — the flat metric dict (also persisted to `metrics.json`) that the figure composer overlays on slice and summary plots.

---
## Stage 9: Figure Composition

**Callable:** `FigureComposer(plotter, meta, logger, cfg).compose(result, run, global_metrics, x_axis_np, indices)`
**Input:** the `Result`, the `Run`, the metric dict, the elevation axis, and the slice indices.
**Output:** a `Dict[str, Path]` mapping figure keys to the `.png` files written under `inference_metadata.figures_dir`.

This stage renders the full diagnostic suite: best / worst / random profile reconstruction panels (pixels selected by per-pixel MSE), per-pixel metric maps (MSE log-scale, $R^2$, peak error), metric histograms, Gaussian parameter spatial maps / distributions / scatter / error maps, slot-$\mu$ statistics, placeholder-detection bars, the slot-ordering summary, the active-Gaussian count map, predicted-vs-GT tomogram slices (range / azimuth / elevation cuts) with SSIM annotations, the dense per-axis SSIM curves, and the per-elevation-bin metric curves.

> **What you should see:** every expected figure key is present in the returned dict and each path exists on disk with non-zero size; the dict includes `profiles_best`, `profiles_worst`, `profiles_random`, `pixel_mse_map`, `pixel_r2_map`, `pixel_peak_map`, `metric_histograms`, the `param_*` family, `active_count_map`, the `ssim_*` curves, and a slice figure per requested slice index.

In [ ]:
figure_composer = FigureComposer(plotter=inference_plotter, meta=inference_metadata, logger=inference_logger, cfg=inference_configuration)

composed_figure_paths = figure_composer.compose(
    result         = inference_result,
    run            = loaded_run,
    global_metrics = global_metrics,
    x_axis_np      = elevation_axis_for_metrics,
    indices        = slice_indices,
)

In [ ]:
print("number of figures        :", len(composed_figure_paths))
print()
missing_figure_files = []
for figure_key in sorted(composed_figure_paths):
    figure_path = composed_figure_paths[figure_key]
    file_exists = figure_path.exists()
    file_size   = figure_path.stat().st_size if file_exists else 0
    if not file_exists or file_size == 0:
        missing_figure_files.append(figure_key)
    print(f"  {figure_key:<26}: exists={file_exists} size={file_size} bytes")
print()
print("missing or empty figures :", missing_figure_files)

In [ ]:
from matplotlib import image as mpl_image

preview_figure_keys = [key for key in ("pixel_r2_map", "pixel_mse_map", "active_count_map", "param_distributions") if key in composed_figure_paths]

figure, axes = plt.subplots(2, 2, figsize=(14, 10))
for axis, preview_key in zip(axes.ravel(), preview_figure_keys):
    preview_path = composed_figure_paths[preview_key]
    try:
        axis.imshow(mpl_image.imread(str(preview_path)))
    except Exception as preview_error:
        axis.text(0.5, 0.5, f"{preview_key}\n({preview_error})", ha="center", va="center")
    axis.set_title(preview_key)
    axis.set_axis_off()
for unused_axis in axes.ravel()[len(preview_figure_keys):]:
    unused_axis.set_axis_off()

figure.suptitle("Stage 9 — Figure Composition: rendered diagnostic previews")
figure.tight_layout(rect=(0, 0, 1, 0.96))
figure.savefig(figure_directory / "stage_9_figures.pdf")
plt.show()

In [ ]:
expected_figure_keys = {
    "profiles_best", "profiles_worst", "profiles_random",
    "pixel_mse_map", "pixel_r2_map", "pixel_peak_map", "metric_histograms",
    "param_maps", "param_distributions", "param_scatter", "param_error_maps",
    "slot_mu_distributions", "placeholder_detection", "slot_ordering_summary", "active_count_map",
    "ssim_range", "ssim_azimuth", "ssim_elev", "elev_metric_curves",
}
expected_slice_figure_count = (
    slice_indices['slice_range_idx'].size
    + slice_indices['slice_az_idx'].size
    + slice_indices['slice_elev_idx'].size
)
rendered_slice_figure_count = sum(1 for key in composed_figure_paths if key.startswith("slice_"))

stage_9_checks = [
    ("compose returned a dict",              isinstance(composed_figure_paths, dict)),
    ("all core figure keys present",         expected_figure_keys.issubset(set(composed_figure_paths))),
    ("every figure file exists",             all(path.exists() for path in composed_figure_paths.values())),
    ("no empty figure files",                all(path.stat().st_size > 0 for path in composed_figure_paths.values() if path.exists())),
    ("slice figure count matches indices",   rendered_slice_figure_count == expected_slice_figure_count),
    ("figures live under figures_dir",       all(inference_metadata.figures_dir in path.parents for path in composed_figure_paths.values())),
]
for stage_9_description, stage_9_condition in stage_9_checks:
    print(f"[{'PASS' if stage_9_condition else 'FAIL'}] {stage_9_description}")

### Common Mistakes — Stage 9: Figure Composition

| Symptom | Likely Cause | How to Diagnose |
|---------|--------------|-----------------|
| Tomogram slice panels look transposed | Pred and GT slice extracted along different axes | Compare against the `axis` argument in `plot_tomogram_slice` |
| Elevation axis upside down in slices | `origin` / sort of `x_axis` inconsistent between pred and GT | Slices sort by `argsort(x_axis)`; both must use the same order |
| Profile panels all empty | `select_pixels` returned no pixels (all metric values non-finite) | Inspect `result.pixel_mse` for all-NaN |
| Parameter scatter shows no active points | Placeholder mask ($a<10^{-3}$) removed every pixel | Check the GT activation rate per slot in `active_count_map` |
| SSIM curve flat at one value | `all_*_idx` slice metrics not computed (dense indices missing) | Stage 8 must receive the `all_*` arrays |
| Colours inconsistent across figures | `Ploter` style not applied / normalization toggled mid-run | Confirm a single `Ploter` instance is reused |

**Passing to Stage 10:** `composed_figure_paths` — the figure-key to path map embedded by the report; the same `Result` and `x_axis` feed the animations next.

---
## Stage 10: Animation Generation

**Callable:** `FigureComposer.animate(result, x_axis_np)` (delegates to `Animator.walk_gif`).
**Input:** the full-scene `Result` and elevation axis.
**Output:** a `Dict[str, Path]` of GIF files (one per axis in `config.gif_axes`), each walking through slices of the stitched cube with GT, prediction, and absolute-error panels side by side under a fixed colour scale.

Frame indices are sampled evenly over the chosen axis up to `config.gif_max_frames`. The colour limits are fixed from a sample of slices so brightness does not flicker between frames; the error panel is clipped at the 99th percentile of $|\hat y - y|$.

> **What you should see:** one GIF path per entry in `config.gif_axes` (default `['elevation']`); each path exists, ends in `.gif`, and has non-zero size; the keys are `walk_<axis>`.

In [ ]:
animation_paths = figure_composer.animate(inference_result, elevation_axis_for_metrics)

In [ ]:
print("requested gif axes       :", inference_configuration.gif_axes)
print("number of animations     :", len(animation_paths))
print()
for animation_key in sorted(animation_paths):
    animation_path = animation_paths[animation_key]
    file_exists    = animation_path.exists()
    file_size      = animation_path.stat().st_size if file_exists else 0
    print(f"  {animation_key:<16}: {animation_path}  exists={file_exists} size={file_size} bytes")

In [ ]:
animation_labels = list(animation_paths.keys())
animation_sizes  = [animation_paths[key].stat().st_size / 1024.0 if animation_paths[key].exists() else 0.0 for key in animation_labels]

figure, axis = plt.subplots(figsize=(7, 3.4))
axis.bar(animation_labels, animation_sizes, color="C0", edgecolor="black", linewidth=0.6)
axis.set_ylabel("GIF file size [KiB]")
axis.set_xlabel("animation key")
axis.set_title("Stage 10 — Animation Generation: rendered GIF sizes")
axis.grid(True, axis="y", linewidth=0.3, alpha=0.4)
figure.tight_layout()
figure.savefig(figure_directory / "stage_10_animations.pdf")
plt.show()

In [ ]:
stage_10_checks = [
    ("animate returned a dict",              isinstance(animation_paths, dict)),
    ("one gif per configured axis",          len(animation_paths) == len(inference_configuration.gif_axes)),
    ("every gif file exists",                all(path.exists() for path in animation_paths.values())),
    ("no empty gif files",                   all(path.stat().st_size > 0 for path in animation_paths.values() if path.exists())),
    ("all keys prefixed walk_",              all(key.startswith("walk_") for key in animation_paths)),
    ("gifs live under animations_dir",       all(inference_metadata.animations_dir in path.parents for path in animation_paths.values())),
]
for stage_10_description, stage_10_condition in stage_10_checks:
    print(f"[{'PASS' if stage_10_condition else 'FAIL'}] {stage_10_description}")

### Common Mistakes — Stage 10: Animation Generation

| Symptom | Likely Cause | How to Diagnose |
|---------|--------------|-----------------|
| Brightness flickers between frames | Per-frame colour scaling instead of a fixed `vmin`/`vmax` | `walk_gif` fixes clim from a sample; verify the sample is non-empty |
| GIF has too few frames | `gif_max_frames` smaller than intended, or axis shorter than expected | Compare requested frames against the axis length |
| Elevation walk shows azimuth/range slices | Wrong `axis` key in `gif_axes` | Valid values are `elevation`, `range`, `azimuth` |
| Rendering hangs / OOM | `gif_workers` too high for available cores / memory | Lower `gif_workers` in the config |
| Error panel saturated | p99 clip computed over a degenerate sample | Inspect `emax_gt`; it falls back to 1.0 when non-positive |

**Passing to Stage 11:** `animation_paths` — the GIF path map, embedded alongside the figures in the final report.

---
## Stage 11: Report Assembly

**Callable:** `InferencePipeline._build_report(...)` -> `Report(...).assemble()`.
**Input:** the run summary, inference config, checkpoint meta, global metrics, figure paths, and gif paths.
**Output:** the path to `report.md`, a self-contained Markdown report embedding the run configuration, headline and full metric tables, and every figure / animation by relative path.

No numerical transformation; this stage serializes the verified artifacts into a human-readable document. Figure links are made relative to the output directory so the report is portable.

> **What you should see:** `report.md` exists under `inference_metadata.output_dir`, is non-empty, contains the headline-metrics and run-summary section headers, and references the figures and animations produced in Stages 9 and 10. The returned value equals `inference_metadata.report_path`.

In [ ]:
report_path = InferencePipeline(inference_configuration)._build_report(
    meta           = inference_metadata,
    run            = loaded_run,
    cfg            = inference_configuration,
    x_axis_np      = elevation_axis_for_metrics,
    global_metrics = global_metrics,
    figure_paths   = composed_figure_paths,
    gif_paths      = animation_paths,
)

In [ ]:
report_text = report_path.read_text(encoding="utf-8")
report_lines = report_text.splitlines()
report_headers = [line for line in report_lines if line.startswith("#")]
embedded_image_count = sum(1 for line in report_lines if line.strip().startswith("!["))

print("report path              :", report_path)
print("report exists            :", report_path.exists())
print("report size (chars)      :", len(report_text))
print("report line count        :", len(report_lines))
print("embedded image / gif refs:", embedded_image_count)
print()
print("section headers:")
for header_line in report_headers:
    print("  ", header_line)

In [ ]:
report_section_labels = ["run\nsummary", "headline\nmetrics", "full metric\ntables", "figures\n+ gifs"]
report_section_counts = [
    sum(1 for line in report_lines if line.startswith("## 1.") or line.startswith("### 1.")),
    sum(1 for line in report_lines if line.startswith("## 2.")),
    sum(1 for line in report_lines if line.startswith("## 3.") or line.startswith("### 3.")),
    embedded_image_count,
]

figure, axis = plt.subplots(figsize=(7.5, 3.6))
axis.bar(report_section_labels, report_section_counts, color="C0", edgecolor="black", linewidth=0.6)
axis.set_ylabel("count (headers or embeds)")
axis.set_xlabel("report section")
axis.set_title("Stage 11 — Report Assembly: section content counts")
axis.grid(True, axis="y", linewidth=0.3, alpha=0.4)
figure.tight_layout()
figure.savefig(figure_directory / "stage_11_report.pdf")
plt.show()

In [ ]:
stage_11_checks = [
    ("report.md exists",                     report_path.exists()),
    ("report path == metadata.report_path",  report_path == inference_metadata.report_path),
    ("report under output_dir",              inference_metadata.output_dir in report_path.parents),
    ("report non-empty",                     len(report_text) > 0),
    ("report has headline-metrics section",  "Headline metrics" in report_text),
    ("report has run-summary section",       "Run summary" in report_text),
    ("report embeds figures or gifs",        embedded_image_count > 0),
    ("report is valid markdown title",       report_lines[0].startswith("# ")),
]
for stage_11_description, stage_11_condition in stage_11_checks:
    print(f"[{'PASS' if stage_11_condition else 'FAIL'}] {stage_11_description}")

### Common Mistakes — Stage 11: Report Assembly

| Symptom | Likely Cause | How to Diagnose |
|---------|--------------|-----------------|
| Broken image links in the rendered report | Figure paths not relative to `output_dir` | `Report._rel` relativizes paths; verify figures live under `output_dir` |
| Report missing figures produced in Stage 9 | A figure key absent from `figure_paths` | The composer dict must reach `_build_report` intact |
| Metric tables show `None` | A metric key renamed in `Metrics` but not in `Report` | Cross-check the key strings between the two modules |
| Report overwrites a prior run's report | Output directory reused (see Stage 1) | Confirm `report_path` is under the intended `output_dir` |
| Numbers formatted inconsistently | Mixed float / scientific formatting | `Report._fmt` switches notation by magnitude; expected behaviour |

**Final handoff:** `report_path` — the `Path` returned by `InferencePipeline.run()`, the single deliverable that bundles every verified artifact.

---
## End-to-End Summary

The stage-by-stage walk above reproduces, in order, exactly what `InferencePipeline.run()` orchestrates. The cell below runs the full pipeline through its single top-level entry point and confirms it returns the same report `Path` and writes the same artifacts the manual walk produced, then prints a consolidated table of the intermediate output shapes, types, and key statistics gathered across all stages.

In [ ]:
end_to_end_configuration = InferenceConfig(
    run_directory = inference_configuration.run_directory,
    output_subdir = "notebook_inspection_end_to_end",
    device        = inference_configuration.device,
    use_ema       = inference_configuration.use_ema,
    split         = inference_configuration.split,
)

end_to_end_report_path = InferencePipeline(end_to_end_configuration).run()
print("end-to-end report path :", end_to_end_report_path)

In [ ]:
summary_rows = [
    ("Stage 1  setup",          "InferenceMetadata",   str(("dirs created",))),
    ("Stage 2  run",            type(loaded_run).__name__, f"K={loaded_run.n_gaussians}, patches={loaded_run.grid.number_of_patches}"),
    ("Stage 4  pred params",    "list[ndarray]",       f"batches={len(forward_pass_pred_params)}, block={np.asarray(forward_pass_pred_params[0]).shape}"),
    ("Stage 5  curves",         "ndarray",             f"first batch curves={first_pred_curves.shape}"),
    ("Stage 6  result cube",    type(inference_result).__name__, f"pred_curves={inference_result.pred_curves.shape}"),
    ("Stage 7  slice indices",  "dict",                f"elev/range/az={slice_indices['slice_elev_idx'].size}/{slice_indices['slice_range_idx'].size}/{slice_indices['slice_az_idx'].size}"),
    ("Stage 8  global metrics", "dict",                f"keys={len(global_metrics)}, R2={global_metrics.get('overall_r2_gt'):.4g}"),
    ("Stage 9  figures",        "dict[str,Path]",      f"figures={len(composed_figure_paths)}"),
    ("Stage 10 animations",     "dict[str,Path]",      f"gifs={len(animation_paths)}"),
    ("Stage 11 report",         type(report_path).__name__, report_path.name),
]

header_format = "{:<24} | {:<20} | {}"
print(header_format.format("Stage / artifact", "Type", "Key statistics"))
print("-" * 96)
for stage_label, artifact_type, key_statistics in summary_rows:
    print(header_format.format(stage_label, artifact_type, key_statistics))
print()

end_to_end_checks = [
    ("end-to-end report exists",             end_to_end_report_path.exists()),
    ("end-to-end report is a markdown file", end_to_end_report_path.suffix == ".md"),
    ("manual walk report exists",            report_path.exists()),
    ("both reports name report.md",          end_to_end_report_path.name == report_path.name == inference_configuration.paths.report_filename),
]
for end_to_end_description, end_to_end_condition in end_to_end_checks:
    print(f"[{'PASS' if end_to_end_condition else 'FAIL'}] {end_to_end_description}")